# Clase 007 — Comprehensions y generadores

**Parte 0** · Ramalho caps. 2 y 17.

> 🎯 Escribir Python idiomático: comprehensions en vez de for+append; generadores para procesar lo que no cabe en RAM.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import sys, time, tracemalloc
from itertools import chain, islice, groupby, accumulate, takewhile
print('python:', sys.version.split()[0])

## 1️⃣ List comprehension

Forma: `[expr for x in iterable if cond]`. Equivale a un `for+append+if`.

```python
# Idiomático
cuadrados_pares = [x*x for x in range(10) if x % 2 == 0]

# Equivalente verboso
result = []
for x in range(10):
    if x % 2 == 0:
        result.append(x*x)
```

In [ ]:
# Tres ejemplos canónicos
cuadrados = [x*x for x in range(10)]
print('cuadrados:', cuadrados)

pares = [x for x in range(20) if x % 2 == 0]
print('pares:', pares)

matriz = [[i*j for j in range(4)] for i in range(4)]
for fila in matriz:
    print(fila)

## 2️⃣ Dict y set comprehensions

Mismo patrón, distinta estructura:

In [ ]:
# Dict comprehension
cuadrados_dict = {x: x*x for x in range(5)}
print(cuadrados_dict)

# Set comprehension (unicidad automática)
letras = {c.lower() for c in 'Hola Mundo' if c.isalpha()}
print(letras)

# Invertir un dict
original = {'a': 1, 'b': 2, 'c': 3}
invertido = {v: k for k, v in original.items()}
print(invertido)

## 3️⃣ Generator expressions — lazy y O(1) memoria

Mismo paréntesis cambiados:

```python
lista     = [x*x for x in range(1_000_000)]    # construye toda la lista en memoria
generador = (x*x for x in range(1_000_000))    # objeto perezoso, calcula on-demand

sum(lista)      # ya está en RAM
sum(generador)  # itera y descarta — RAM O(1)
```

In [ ]:
# Medición real de memoria
def midiendo(label, fn):
    tracemalloc.start()
    fn()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f'{label:20s} peak={peak/1024:.1f} KB')

N = 1_000_000
midiendo('lista', lambda: sum([x*x for x in range(N)]))
midiendo('generador', lambda: sum(x*x for x in range(N)))

## 4️⃣ Funciones generadoras con `yield`

Cualquier función con `yield` se convierte en generador. Cada `yield` pausa y emite un valor; la siguiente iteración resume.

In [ ]:
def fibonacci():
    """Generador infinito de Fibonacci."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Toma los primeros 10 sin construir la lista infinita
primeros_10 = list(islice(fibonacci(), 10))
print(primeros_10)

# Toma mientras sean < 1000
fibs_chicos = list(takewhile(lambda x: x < 1000, fibonacci()))
print(fibs_chicos)

## 5️⃣ `itertools` — la navaja suiza

- `chain(a, b, c)` — concatena iterables sin allocar lista
- `islice(it, start, stop, step)` — slicing perezoso
- `groupby(it, key)` — agrupa elementos *consecutivos* con misma key
- `accumulate(it, fn)` — suma/producto acumulado
- `combinations(it, r)` / `permutations(it, r)` — combinatoria

In [ ]:
# Demo: groupby — atención, agrupa CONSECUTIVOS
datos = sorted([('a', 1), ('a', 2), ('b', 3), ('a', 4)], key=lambda x: x[0])
for k, grupo in groupby(datos, key=lambda x: x[0]):
    print(k, list(grupo))

print()

# accumulate: suma acumulada
print(list(accumulate([1, 2, 3, 4, 5])))  # [1, 3, 6, 10, 15]

## 6️⃣ Cuándo NO usar comprehension

- Lógica de más de 2 líneas → for explícito
- Side effects (`print`, mutación) → for explícito
- Anidamiento triple → for explícito
- Cuando el reviewer no la entiende → for explícito

Regla: comprehension es **expresión**, no **statement**. Si necesitas múltiples statements, usa for.

## ✅ Checklist

- [ ] Sé convertir `for+append` a comprehension
- [ ] Distingo `[...]` (lista) de `(...)` (generador)
- [ ] Sé escribir un generador con `yield`
- [ ] Conozco `chain`, `islice`, `groupby` de itertools
- [ ] Sé cuándo NO usar comprehension

## 📝 Homework

Ver `README.md`. Reescribir 3 loops, generador Fibonacci, medir RAM lista vs generador, procesar CSV grande con generador.

## 📖 Definiciones y características

**List comprehension**

Expresión `[expr for x in iterable if cond]` que **construye una lista** completa en memoria. Equivalente idiomático a `for + append + if`. Más rápida y legible *si la expresión es simple*.

**Generator expression**

Lo mismo pero con paréntesis `(expr for x in iterable if cond)`. Devuelve un **generador perezoso**: produce cada valor on-demand, memoria O(1). Solo puedes recorrerlo una vez.

**Iterador**

Objeto con método `__next__()` que devuelve el siguiente valor o lanza `StopIteration`. Es lo que está *detrás* de un for. Características: lazy, single-pass, memoria O(1).

**Generador**

Función con `yield` (o generator expression) que produce un iterador. Cada `yield` pausa la función y emite un valor; al siguiente next() retoma desde ahí. Mantiene el estado local entre llamadas.

**`itertools`**

Librería stdlib con bloques perezosos componibles: `chain` (concatena), `islice` (slicing perezoso), `groupby` (agrupa consecutivos), `accumulate` (sum/prod corridos), `takewhile`, `combinations`, `product`.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Iteré un generador y la segunda vez está vacío | Los generadores son **single-pass**. Tras consumir, están agotados. **Fix**: convierte a lista (`list(gen)`) si necesitas recorrer más veces — pierdes memoria O(1). |
| `MemoryError` al hacer `[expensive(x) for x in millones]` | Construyes lista completa en RAM. **Fix**: usa generator expression `(expensive(x) for x in millones)` y consúmelo perezosamente con `sum()`, `next()`, `for`. |
| `itertools.groupby` me agrupa raro | Solo agrupa **elementos consecutivos** con misma key. Si los datos no están ordenados por la key, primero `sorted(data, key=...)`. |
| Generator expression dentro de función falla con `'generator' object is not subscriptable` | Estás haciendo `gen[0]` — generadores no son indexables. **Fix**: `next(gen)` para el primer valor, o `list(gen)[0]` si necesitas acceso aleatorio. |
| List comprehension con `if-else` no funciona como espero | `if` al final filtra; `if-else` va al principio: `[x if x > 0 else 0 for x in nums]` (clip). NO `[x for x in nums if x > 0 else 0]` (syntax error). |

## ❓ Preguntas frecuentes

**❓ ¿Cuándo comprehension y cuándo for clásico?**

Comprehension cuando es **una expresión** clara en 1 línea. For clásico cuando hay >2 statements, side effects (print, mutación), o la lógica es más legible explícita.

**❓ ¿Generator expression o list?**

Generator si el resultado solo se consume una vez y N es grande (RAM importa). List si necesitas recorrer 2+ veces, indexar, o hacer `len()`. Truco: `sum(x*x for x in xs)` evita lista temporal vs `sum([x*x for x in xs])`.

**❓ ¿Cuánto más rápido es vs un for tradicional?**

~10-30%, no mil veces más. La ganancia real es legibilidad. Si necesitas mil veces, no es comprehension lo que buscas — es numpy/vectorización (clase 015).

**❓ ¿Generador infinito (`while True: yield ...`) es buena idea?**

Sí, pero ojo: **nunca** lo conviertas a lista directo (`list(gen)`) — bucle infinito hasta OOM. Acopla con `itertools.islice(gen, N)` para truncar.

**❓ `yield from` vs `yield`?**

`yield from sub_iterable` delega: yieldea todos los valores del sub-iterable. Equivale a `for x in sub: yield x` pero más rápido y propaga `send()`/`throw()` correctamente. Útil para componer generadores.

## 🔗 Referencias

- Ramalho, *Fluent Python* 2e — caps. 2 y 17
- [itertools docs](https://docs.python.org/3/library/itertools.html)

➡️ **Siguiente:** [008 — Funciones: args, kwargs, lambdas, closures](../008-funciones-args-kwargs-lambdas-closures/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá está una solución de referencia comentada.

**Ejercicio 1.** De for a comprehension: convertir 3 loops `for+append` (cuadrados, filtrar pares, mapear a strings).

In [ ]:
# Solución Ej.1 — cada loop imperativo -> su comprehension equivalente.
nums = list(range(1, 11))

# a) cuadrados
cuad_loop = []
for n in nums:
    cuad_loop.append(n * n)
cuad = [n * n for n in nums]
assert cuad == cuad_loop == [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

# b) filtrar pares
pares_loop = []
for n in nums:
    if n % 2 == 0:
        pares_loop.append(n)
pares = [n for n in nums if n % 2 == 0]
assert pares == pares_loop == [2, 4, 6, 8, 10]

# c) mapear a strings
strs = [f"#{n}" for n in nums]
assert strs[:3] == ["#1", "#2", "#3"]

print("cuadrados:", cuad)
print("pares:", pares)
print("strings:", strs[:3], "...")
print("OK: los 3 loops reescritos como comprehensions.")

**Ejercicio 2.** Generador de Fibonacci infinito con `yield`, truncado con `itertools.islice` a los primeros 20.

In [ ]:
# Solución Ej.2 — generador infinito + islice para truncar (nunca list(gen) directo!).
import itertools

def fib():
    a, b = 0, 1
    while True:
        yield a          # pausa aquí y emite; retoma en el próximo next()
        a, b = b, a + b

primeros_20 = list(itertools.islice(fib(), 20))
print(primeros_20)
assert primeros_20[:8] == [0, 1, 1, 2, 3, 5, 8, 13]
assert len(primeros_20) == 20
assert primeros_20[-1] == 4181
print("OK: Fibonacci infinito truncado con islice.")

**Ejercicio 3.** Memoria: medir RAM (con `tracemalloc`) de una list comprehension vs un generator expression y reportar la diferencia.

In [ ]:
# Solución Ej.3 — comparar el pico de memoria de construir una lista vs consumir un generador.
# Usamos un N moderado para correr rápido en CI; el efecto es el mismo a cualquier escala.
import tracemalloc

N = 1_000_000

# a) List comprehension: materializa TODOS los cuadrados en RAM antes de sumar.
tracemalloc.start()
total_lista = sum([i * i for i in range(N)])
pico_lista = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

# b) Generator expression: produce un cuadrado a la vez, memoria ~O(1).
tracemalloc.start()
total_gen = sum(i * i for i in range(N))
pico_gen = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

assert total_lista == total_gen          # mismo resultado numérico
print(f"pico lista     : {pico_lista:,} bytes")
print(f"pico generador : {pico_gen:,} bytes")
print(f"la lista usó ~{pico_lista / max(pico_gen, 1):.0f}x más memoria")
assert pico_gen < pico_lista             # el generador SIEMPRE gana en memoria
print("OK: el generator expression evita materializar la lista completa.")

**Ejercicio 4.** Procesar un archivo grande línea por línea con `yield`, filtrar por una condición y contar, sin cargar todo en memoria.

In [ ]:
# Solución Ej.4 — pipeline perezoso sobre un CSV. Generamos un CSV temporal para correr sin red.
import tempfile, pathlib, csv

def leer_lineas(ruta):
    """Yieldea una línea a la vez: la memoria no crece con el tamaño del archivo."""
    with open(ruta, encoding="utf-8") as fh:
        header = fh.readline()          # descarta cabecera
        for linea in fh:                # el for sobre un file object ya es perezoso
            yield linea.rstrip("\n")

with tempfile.TemporaryDirectory() as d:
    ruta = pathlib.Path(d) / "ventas.csv"
    with open(ruta, "w", encoding="utf-8", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["id", "monto"])
        for i in range(10_000):
            w.writerow([i, i % 100])    # montos 0..99 cíclicos

    # contar filas con monto > 50 SIN cargar el archivo entero
    n_altos = sum(1 for linea in leer_lineas(ruta) if int(linea.split(",")[1]) > 50)
    print("filas con monto > 50:", n_altos)
    # montos 51..99 aparecen 100 veces cada uno en 10_000 filas -> 49*100 = 4900
    assert n_altos == 4900
print("OK: CSV procesado línea a línea, sin materializarlo en RAM.")

**Ejercicio 5.** Pivot con dict comprehension: dada `list[tuple[str, int]]` (nombre, puntaje), construir `dict[str, list[int]]` agrupando puntajes por nombre.

In [ ]:
# Solución Ej.5 — pivot nombre -> lista de puntajes.
# El agrupamiento necesita acumular, así que primero agrupamos y luego la dict comprehension
# ordena/normaliza. (Una dict comprehension pura no acumula; por eso el setdefault previo.)
datos = [("ana", 10), ("beto", 7), ("ana", 8), ("beto", 9), ("ana", 5)]

agrupado = {}
for nombre, puntaje in datos:
    agrupado.setdefault(nombre, []).append(puntaje)

# dict comprehension para, por ejemplo, ordenar cada lista de puntajes desc
pivot = {nombre: sorted(puntajes, reverse=True) for nombre, puntajes in agrupado.items()}
print(pivot)
assert pivot == {"ana": [10, 8, 5], "beto": [9, 7]}
# y una dict comprehension para el promedio por nombre
promedios = {nombre: sum(v) / len(v) for nombre, v in pivot.items()}
assert promedios["ana"] == (10 + 8 + 5) / 3
print("promedios:", promedios)
print("OK: pivot y agregación con dict comprehension.")